# 02 — Tile-Based Litter Classification (Custom CNN from scratch)

**Project:** CNN-Based Aerial Litter Detection for Sustainable Trail and Environmental Cleanup  
**Module:** ST7088CEM Artificial Neural Networks

Task 1: aerial images are sliced into 512 px tiles labelled litter/no-litter from
annotation overlap, and a custom CNN (stacked conv–pool blocks + dense head) is
designed and trained **from scratch**. Class imbalance (~15% positive) is handled
with a weighted BCE loss; training uses an LR schedule and early stopping.
Metrics: accuracy, precision, recall, F1, confusion matrix, learning curves.

## 1. Environment setup

On **Google Colab / Kaggle** (GPU runtime recommended): uncomment the clone +
install lines, and the Drive lines if the dataset is stored there.

In [ ]:
# On Colab/Kaggle, uncomment:
# !git clone https://github.com/Sajan491/STW7088CEM-ANN-Assignment.git
# %cd STW7088CEM-ANN-Assignment
# %pip install -q -r requirements.txt

# On Colab, if the dataset lives in Google Drive, also uncomment:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/UAVVaste/data ./data

import os, sys, platform
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir('..')
sys.path.insert(0, str(Path.cwd()))

import torch
print('Working directory:', Path.cwd())
print('Python:', sys.version)
print('Machine:', platform.node(), '|', platform.platform())
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Tile dataset

Requires the dataset under `data/` (see README). This cell generates the splits
and the ~28k labelled tile crops if they are not already present (a few minutes).

In [ ]:
if not Path('data/processed/splits.json').exists():
    !python -m src.data.splits
if not Path('data/processed/tiles/train').exists():
    !python -m src.data.tiles
else:
    print('tile crops already present — skipping generation')

for split in ['train', 'val', 'test']:
    pos = len(list(Path(f'data/processed/tiles/{split}/pos').glob('*.jpg')))
    neg = len(list(Path(f'data/processed/tiles/{split}/neg').glob('*.jpg')))
    print(f'{split}: {pos} pos / {neg} neg ({pos / max(pos + neg, 1):.1%} positive)')

## 3. Sample tiles

A quick visual check of what the network sees: litter tiles (top row) vs
no-litter tiles (bottom row).

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, p in zip(axes[0], sorted(Path('data/processed/tiles/train/pos').glob('*.jpg'))[:5]):
    ax.imshow(Image.open(p)); ax.set_title('litter', fontsize=10); ax.axis('off')
for ax, p in zip(axes[1], sorted(Path('data/processed/tiles/train/neg').glob('*.jpg'))[:5]):
    ax.imshow(Image.open(p)); ax.set_title('no litter', fontsize=10); ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Train the custom CNN

Configuration lives in `configs/tile_classifier.yaml` (architecture, weighted
loss, ReduceLROnPlateau schedule, early stopping on validation F1). The best
checkpoint is saved to `checkpoints/tile_cnn_best.pt`; per-epoch history to
`results/tables/tile_training_history.csv`.

In [ ]:
!python -m src.training.train_tile_classifier

## 5. Evaluate on the held-out test split

Reports accuracy, precision, recall, F1 and ROC-AUC, and saves the confusion
matrix and learning-curve figures to `results/figures/`.

In [ ]:
!python -m src.evaluation.evaluate_tile_classifier

## 6. Results

In [ ]:
import pandas as pd
from IPython.display import Image as IPImage, display

display(pd.read_csv('results/tables/tile_classifier_metrics.csv'))
display(IPImage('results/figures/tile_learning_curves.png', width=900))
display(IPImage('results/figures/tile_confusion_matrix.png', width=520))

## 7. Notes for the report

- The custom CNN is trained **from scratch** — no pretrained weights — so the
  learning curves directly show what the architecture learns from ~20k tiles.
- The weighted BCE loss (pos_weight ≈ 5.6) counteracts the ~15% positive rate;
  compare precision vs recall in the confusion matrix when discussing it.
- Early stopping + ReduceLROnPlateau keep the model at its best validation F1
  epoch; the reported test metrics come from that checkpoint.
- Splits are image-level (Phase 1), so no tile from a training image can leak
  into validation or test.